# Manuscript revision diagnostics — Ca Mau

Sections follow Phase 3 of the revision plan; all use the existing posterior at `data/outputs/camau/r1202/calib_full_out.nc`. No re-calibration.

1. Per-parameter ESS / R-hat → `outputs/calibration_diagnostics.csv`
2. Posterior corner plot → `outputs/posterior_corner.png`
3. Posterior correlation matrix → `outputs/posterior_correlations.csv`
4. Posterior predictive checks → `outputs/ppc_*.png`, `outputs/ppc_summary.csv`
5. Recent transmission share time-series → `outputs/recent_transmission_share.csv`
6. Residual incidence under sustained biennial ACF in 2035

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
from estival.sampling import tools as esamp
from tbdynamics.camau.calibration.utils import (
    get_bcm, calculate_future_acf_outputs,
)
from tbdynamics.camau.constants import params_name
from tbdynamics.tools.detect import make_future_acf_scenarios
from tbdynamics.calibration.az_aux import (
    report_calibration_diagnostics,
    plot_posterior_corner,
    compute_posterior_correlations,
    run_ppc,
    run_indicators_for_samples,
    plot_ppc_panel,
    compute_recent_transmission_metrics,
    quantile_summary,
)

## Setup

In [ ]:
RUN_PATH = Path.cwd().parent.parent / 'data/outputs/camau/r1202'
OUT_DIR = Path.cwd().parent.parent / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
BURN_IN = 100_000          # adjust if your chain is shorter / longer
N_PPC_SAMPLES = 1_000      # number of samples for PPC / time-series posterior runs
RANDOM_STATE = 42

idata = az.from_netcdf(RUN_PATH / 'calib_full_out.nc')
print('Posterior dims:', dict(idata.posterior.sizes))
print('Variables:', list(idata.posterior.data_vars))

In [ ]:
burnt = idata.sel(draw=slice(BURN_IN, None))
print('Post burn-in dims:', dict(burnt.posterior.sizes))
burnt

In [ ]:
# Match the params/covid_effects used at calibration time
params = {
    'seed_num': 1.0,
    'seed_duration': 1.0,
    'screening_scaleup_shape': 0.3,
    'screening_inflection_time': 1993,
}
covid_effects = {'detection_reduction': True, 'contact_reduction': False}
bcm = get_bcm(params, covid_effects)
list(bcm.targets)

In [ ]:
# Sub-sample for computational sections (3.4, 3.5)
idata_extract = az.extract(burnt, num_samples=N_PPC_SAMPLES, rng=RANDOM_STATE)
idata_extract = az.convert_to_inference_data(idata_extract.reset_index('sample'))
idata_extract

## 3.1 Per-parameter ESS and R-hat

In [ ]:
diag_table, flagged = report_calibration_diagnostics(
    burnt, params_name=params_name, ess_threshold=400, rhat_threshold=1.05,
)
diag_table

In [ ]:
if flagged:
    print('FLAGGED parameters (ESS<400 or R-hat>1.05):')
    for p in flagged:
        print('  -', p)
else:
    print('All parameters meet ESS>=400 and R-hat<=1.05.')

diag_table.to_csv(OUT_DIR / 'calibration_diagnostics.csv')

# Markdown summary for manuscript supplement (built without `tabulate`)
def _df_to_md(df):
    cols = [df.index.name or ''] + list(df.columns)
    lines = ['| ' + ' | '.join(map(str, cols)) + ' |',
             '|' + '|'.join(['---'] * len(cols)) + '|']
    for idx, row in df.iterrows():
        cells = [str(idx)] + [f'{v}' for v in row.values]
        lines.append('| ' + ' | '.join(cells) + ' |')
    return '\n'.join(lines)

md_lines = ['# Calibration diagnostics summary', '',
            f'- Total parameters: {len(diag_table)}',
            f'- Min ESS bulk: {diag_table["ess_bulk"].astype(float).min():.0f}',
            f'- Max R-hat: {diag_table["r_hat"].astype(float).max():.3f}',
            f'- Flagged: {flagged if flagged else "none"}', '',
            _df_to_md(diag_table)]
(OUT_DIR / 'calibration_diagnostics.md').write_text('\n'.join(md_lines), encoding='utf-8')
print(f'Saved {OUT_DIR / "calibration_diagnostics.csv"}')
print(f'Saved {OUT_DIR / "calibration_diagnostics.md"}')

## 3.2 Posterior corner plot

In [ ]:
fig_corner = plot_posterior_corner(burnt, exclude=None, params_name=params_name, figsize=(22, 22))
fig_corner.savefig(OUT_DIR / 'posterior_corner.png', dpi=200, bbox_inches='tight')
print(f'Saved {OUT_DIR / "posterior_corner.png"}')
fig_corner

## 3.3 Posterior correlation matrix

Pairs with $|\rho| > 0.7$ are flagged for the Limitations section.

In [ ]:
corr, high_pairs = compute_posterior_correlations(burnt, exclude=None, threshold=0.7)
corr.to_csv(OUT_DIR / 'posterior_correlations.csv')
print(f'Saved {OUT_DIR / "posterior_correlations.csv"}')

if high_pairs.empty:
    print('No parameter pairs with |rho| > 0.7.')
else:
    print('High-correlation pairs (|rho| > 0.7):')
    display(high_pairs)

fig_corr, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticklabels(corr.index, fontsize=8)
fig_corr.colorbar(im, ax=ax, fraction=0.04)
fig_corr.tight_layout()
fig_corr.savefig(OUT_DIR / 'posterior_correlations.png', dpi=180, bbox_inches='tight')
fig_corr

## 3.4 Posterior predictive checks

For each calibration target, compute the distribution of predicted values at observed time-points and the Bayesian p-value $P(\text{simulated} > \text{observed})$.

In [ ]:
target_names = list(bcm.targets)
predicted = run_ppc(idata_extract, bcm, target_names)
for t, df in predicted.items():
    print(f'{t}: shape {df.shape} (rows=time, cols=samples)')

In [ ]:
fig_ppc, ppc_summary = plot_ppc_panel(predicted, bcm, n_cols=2)
fig_ppc.savefig(OUT_DIR / 'ppc_panel.png', dpi=200, bbox_inches='tight')
ppc_summary.to_csv(OUT_DIR / 'ppc_summary.csv', index=False)

# Bayesian p-values close to 0 or 1 indicate poor calibration
extreme = ppc_summary[(ppc_summary.bayesian_p_value < 0.05) | (ppc_summary.bayesian_p_value > 0.95)]
if not extreme.empty:
    print('Time-points with extreme Bayesian p-values (<0.05 or >0.95):')
    display(extreme)
else:
    print('All Bayesian p-values are in [0.05, 0.95].')
fig_ppc

## 3.5 Recent-transmission share time-series

Run the model on the posterior subset, extract recent-transmission-share indicators (overall + per ACT3 arm), and compute summary metrics:
- baseline (2013), minimum during ACT3 (2014–2018)
- value at 2 and 5 years post-ACT3
- time to rebound (years until recent share returns to within 5 percentage points of baseline)

In [ ]:
indicators = ['incidence_early_perc',
              'recent_infection_percXact3_trial',
              'recent_infection_percXact3_control',
              'recent_infection_percXact3_other']
ts_results = run_indicators_for_samples(idata_extract, bcm, indicators, batch_size=100)
available = list(ts_results)
print('Available indicators:', available)

In [ ]:
ts_quantiles = {}
metrics_per_arm = {}
for ind in available:
    df = ts_results[ind]              # rows=time, cols=sample
    qs = df.quantile([0.025, 0.5, 0.975], axis=1).T
    qs.columns = ['q2.5', 'median', 'q97.5']
    ts_quantiles[ind] = qs
    m = compute_recent_transmission_metrics(df, indicator=ind)
    metrics_per_arm[ind] = {k: quantile_summary(v) for k, v in m.items()}

In [ ]:
# Save quantile time-series CSV
out = pd.concat({k: v for k, v in ts_quantiles.items()}, axis=1)
out.to_csv(OUT_DIR / 'recent_transmission_share.csv')
print(f'Saved {OUT_DIR / "recent_transmission_share.csv"}')

# Plot
fig_rt, axes = plt.subplots(len(available), 1, figsize=(11, 3.2 * len(available)), sharex=True)
if len(available) == 1:
    axes = [axes]
for ax, ind in zip(axes, available):
    qs = ts_quantiles[ind]
    ax.fill_between(qs.index, qs['q2.5'], qs['q97.5'], alpha=0.25)
    ax.plot(qs.index, qs['median'], lw=1.4)
    ax.axvspan(2014, 2018, color='orange', alpha=0.15, label='ACT3 trial')
    ax.axvline(2013, color='grey', ls='--', lw=0.8)
    ax.set_title(ind)
    ax.set_ylabel('% recent transmission')
    ax.legend(fontsize=8, loc='upper right')
axes[-1].set_xlabel('Year')
fig_rt.tight_layout()
fig_rt.savefig(OUT_DIR / 'recent_transmission_share.png', dpi=200, bbox_inches='tight')
fig_rt

In [ ]:
# Single-table summary of recent-transmission metrics
rows = []
for ind, ms in metrics_per_arm.items():
    for metric_name, qs in ms.items():
        rows.append({'indicator': ind, 'metric': metric_name, **qs})
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(OUT_DIR / 'recent_transmission_metrics.csv', index=False)
metrics_df

## 3.6 Residual incidence under sustained biennial ACF in 2035

Use the existing biennial ACF scenario builder, take the median (2.5%, 97.5%) projected incidence per 100,000 in 2035.

In [ ]:
# Pick the biennial / 80% coverage scenario (adjust if you want a different one)
all_scenarios = make_future_acf_scenarios()
# Filter to biennial-only — keys look like 'all_2_50', 'all_2_80', 'all_4_50', 'all_4_80'
biennial_keys = [k for k in all_scenarios if '_2_' in k]
print('Biennial scenarios available:', biennial_keys)
selected_scenarios = {k: all_scenarios[k] for k in biennial_keys}

In [ ]:
future_outputs = calculate_future_acf_outputs(
    params=params,
    idata_extract=idata_extract,
    covid_effects=covid_effects,
    future_acf_scenarios=selected_scenarios,
    request_outputs=['incidence', 'mortality'],
)

In [ ]:
# Extract incidence quantiles at 2035 for each biennial scenario
rows = []
for sname, qdf in future_outputs.items():
    inc = qdf['incidence']
    # nearest available year to 2035
    idx = inc.index.get_indexer([2035.0], method='nearest')[0]
    row = inc.iloc[idx]
    rows.append({'scenario': sname, 'year': float(inc.index[idx]),
                 **{f'q{int(c*100):02d}': row[c] for c in row.index}})
incidence_2035 = pd.DataFrame(rows)
incidence_2035.to_csv(OUT_DIR / 'incidence_2035_biennial_acf.csv', index=False)
incidence_2035

In [ ]:
# Single-value summary for Results section 6.5: median (95% CrI) incidence/100k in 2035 under biennial 80% ACF
key = 'all_2_80'
if key in future_outputs:
    inc = future_outputs[key]['incidence']
    idx = inc.index.get_indexer([2035.0], method='nearest')[0]
    row = inc.iloc[idx]
    median = row.get(0.5, row.iloc[len(row) // 2])
    lo = row.get(0.025, row.iloc[0])
    hi = row.get(0.975, row.iloc[-1])
    print(f'Residual TB incidence in {float(inc.index[idx]):.0f} under biennial 80% ACF:')
    print(f'  median = {median:.1f} per 100,000 (95% CrI {lo:.1f} – {hi:.1f})')
else:
    print(f'Scenario {key} not found.')